# Academic Risk Prediction — Complete Reproducible Analysis

এই notebook-টি আপনার paper-এর analysis এক জায়গায় চালানোর জন্য তৈরি। এটি CSV load, duplicate/answer-pattern audit, group-aware validation, একই fold-এ baseline বনাম tuned-threshold comparison, uncertainty summaries, permutation importance, এবং paper-ready CSV/figure তৈরি করবে।

## শুরু করার আগে
1. মূল Kaggle dataset-এর CSV ফাইলটি এই notebook-এর পাশে রাখুন অথবা `DATA_PATH`-এ তার path দিন।
2. `TARGET_COLUMN`-এ CGPA column-এর সঠিক নাম দিন, যদি auto-detection ভুল হয়।
3. প্রথমে `QUICK_RUN = True` দিয়ে pipeline পরীক্ষা করুন। Final paper numbers-এর জন্য `QUICK_RUN = False` করে সব cell আবার চালান।
4. সব output `paper_outputs/` folder-এ লেখা হবে।

**গুরুত্বপূর্ণ:** Notebook কোনো metric বা ফলাফল বানিয়ে বসায় না। আপনার আসল CSV ছাড়া final numbers পাওয়া যাবে না। উত্তর-pattern একই হলেই সেটি নিশ্চিতভাবে একই ব্যক্তি—এমন দাবি করা হবে না। এই analysis-এ group = একই predictor-answer pattern।

## 1. Configuration

In [ ]:
from pathlib import Path
import os, re, json, warnings, sys, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score,
    recall_score, f1_score, accuracy_score, confusion_matrix,
    brier_score_loss
)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")

# ---- EDIT THESE IF NEEDED ----
DATA_PATH = None   # None = upload CSV interactively below; or set a path like "/content/student_gaming.csv"
TARGET_COLUMN = None               # e.g. "CGPA"; None attempts automatic detection
DROP_COLUMNS = []                  # IDs/timestamps or non-predictor columns, e.g. ["Timestamp"]
CGPA_CUTOFF = 3.0                  # at-risk = CGPA < 3.0
N_SPLITS = 5
QUICK_RUN = True                   # True = smoke test; False = longer final run
OUTPUT_DIR = Path("paper_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if QUICK_RUN:
    SEEDS = [0, 1]
    N_TREES = 100
    N_PERM_REPEATS = 2
    N_BOOTSTRAP = 300
else:
    SEEDS = [0, 1, 2, 3, 4]
    N_TREES = 500
    N_PERM_REPEATS = 10
    N_BOOTSTRAP = 2000

print("Python:", sys.version.split()[0], "| platform:", platform.platform())
print("Output directory:", OUTPUT_DIR.resolve())
print("Quick run:", QUICK_RUN)

## 2A. Upload the dataset (Google Colab)

Run this cell and choose the original CSV dataset from your computer. The upload happens only when you select the file. If you already saved the CSV in the Colab runtime, set `DATA_PATH` manually in Section 1 and skip this upload cell.

In [ ]:
# Google Colab: upload the CSV from your computer
if DATA_PATH is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_names = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
        if not csv_names:
            raise FileNotFoundError("No CSV file was selected. Please upload the original dataset as a .csv file.")
        if len(csv_names) > 1:
            print("Multiple CSVs uploaded:", csv_names)
            print("Using the first file:", csv_names[0])
        DATA_PATH = csv_names[0]
        print("Dataset selected:", DATA_PATH)
    except ImportError:
        raise RuntimeError("This upload cell is for Google Colab. In local Jupyter, set DATA_PATH to the CSV file path in Section 1.")
else:
    print("Using configured DATA_PATH:", DATA_PATH)


## 2. Load CSV and identify target

Auto-detection looks for a column name containing `cgpa`. Please inspect the printed column list and target distribution before continuing. If CGPA values contain categories/text, edit the target cleaning cell as needed rather than accepting a silent assumption.

In [ ]:
if not Path(DATA_PATH).exists():
    candidates = list(Path(".").glob("*.csv"))
    print("CSV files in current directory:", [str(p) for p in candidates])
    raise FileNotFoundError(
        f"Cannot find {DATA_PATH!r}. Put the dataset beside this notebook or update DATA_PATH in Section 1."
    )

df_raw = pd.read_csv(DATA_PATH)
df_raw.columns = [str(c).strip() for c in df_raw.columns]
print("Raw shape:", df_raw.shape)
print("Columns:")
for c in df_raw.columns:
    print(" •", repr(c))

if TARGET_COLUMN is None:
    matches = [c for c in df_raw.columns if "cgpa" in c.lower()]
    if not matches:
        raise ValueError("No column name containing 'cgpa' found. Set TARGET_COLUMN manually in Section 1.")
    target_col = matches[0]
else:
    target_col = TARGET_COLUMN
    if target_col not in df_raw.columns:
        raise ValueError(f"TARGET_COLUMN={target_col!r} not found. Available columns: {df_raw.columns.tolist()}")

print("\nSelected target column:", target_col)
display(df_raw.head())

## 3. Target encoding, predictor selection, and data audit

Target is encoded as `1` when numeric CGPA is below 3.00, otherwise `0`. Non-numeric/missing CGPA rows are excluded from modelling and counted. Group IDs are based on the predictor-answer pattern only, before one-hot encoding. The audit separately reports exact duplicate feature+label rows, duplicate predictor patterns, and patterns associated with conflicting labels.

In [ ]:
df = df_raw.copy()
cgpa_numeric = pd.to_numeric(df[target_col], errors="coerce")
valid_target = cgpa_numeric.notna()
excluded_target_rows = int((~valid_target).sum())
df = df.loc[valid_target].copy()
cgpa_numeric = cgpa_numeric.loc[valid_target]
y = (cgpa_numeric < CGPA_CUTOFF).astype(int).reset_index(drop=True)

drop_set = set(DROP_COLUMNS)
drop_set.add(target_col)
# common identifier-like columns are excluded by default; add others explicitly to DROP_COLUMNS
for c in df.columns:
    cl = c.strip().lower()
    if cl in {"id", "timestamp", "email", "email address", "name", "student id", "serial", "serial no", "respondent id"}:
        drop_set.add(c)

X = df.drop(columns=[c for c in drop_set if c in df.columns]).reset_index(drop=True)
df_model = df.reset_index(drop=True)

if X.shape[1] == 0:
    raise ValueError("No predictors remain. Check DROP_COLUMNS and target configuration.")

# Normalize missing-value representation for stable answer-pattern grouping
X_audit = X.copy()
X_audit = X_audit.astype("object").where(pd.notna(X_audit), "__MISSING__")
row_keys = pd.util.hash_pandas_object(X_audit, index=False).astype("uint64")
groups = pd.factorize(row_keys, sort=False)[0]

# Exact duplicates when predictors and label are both considered
combined = X_audit.copy()
combined["__target__"] = y.to_numpy()
n_rows = len(combined)
n_unique_feature_label = int(combined.drop_duplicates().shape[0])
exact_duplicate_excess = n_rows - n_unique_feature_label

# Distinct predictor patterns and conflicting labels
audit_frame = X_audit.copy()
audit_frame["__target__"] = y.to_numpy()
pattern_stats = audit_frame.groupby(list(X_audit.columns), dropna=False, sort=False)["__target__"].agg(["size", "nunique", "mean"]).reset_index()
n_patterns = len(pattern_stats)
conflict_patterns = pattern_stats[pattern_stats["nunique"] > 1]
conflict_rows = int(conflict_patterns["size"].sum())

audit = pd.DataFrame([
    ["Rows in source CSV", len(df_raw)],
    ["Rows excluded because target is missing/non-numeric", excluded_target_rows],
    ["Rows used in modelling", len(X)],
    ["Number of predictors", X.shape[1]],
    ["At-risk rows (CGPA < cutoff)", int(y.sum())],
    ["At-risk prevalence", float(y.mean())],
    ["Distinct predictor-answer patterns / groups", n_patterns],
    ["Exact duplicate excess rows (predictors + label)", exact_duplicate_excess],
    ["Distinct predictor+label records", n_unique_feature_label],
    ["Conflicting predictor patterns", len(conflict_patterns)],
    ["Rows belonging to conflicting patterns", conflict_rows],
], columns=["Audit item", "Value"])

display(audit)
audit.to_csv(OUTPUT_DIR / "01_data_audit.csv", index=False)
pattern_stats.to_csv(OUTPUT_DIR / "02_answer_pattern_audit.csv", index=False)

print("\nPredictor columns:")
print(X.columns.tolist())
print("\nTarget distribution:")
display(y.value_counts().rename(index={0:"Not at risk",1:"At risk"}).to_frame("count").assign(proportion=lambda z: z["count"]/len(y)))

## 4. Preprocessing and model definitions

Imputation, scaling, and one-hot encoding are fitted inside each training fold to reduce leakage. We use the same outer folds and same predicted probabilities for threshold 0.50 and tuned-threshold results. Therefore ROC-AUC and PR-AUC must be identical between those two rows for each model/seed; threshold tuning can change precision, recall, F1, and flagged fraction only.

In [ ]:
numeric_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError:  # compatibility with older scikit-learn
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=True)

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False))
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", ohe)
])
transformers = []
if numeric_cols:
    transformers.append(("num", numeric_pipe, numeric_cols))
if categorical_cols:
    transformers.append(("cat", categorical_pipe, categorical_cols))
preprocessor = ColumnTransformer(transformers=transformers, remainder="drop", sparse_threshold=0.0)

def make_models(seed):
    return {
        "Random Forest": RandomForestClassifier(
            n_estimators=N_TREES, max_features="sqrt", class_weight=None,
            random_state=seed, n_jobs=-1
        ),
        "Extra Trees": ExtraTreesClassifier(
            n_estimators=N_TREES, max_features="sqrt", class_weight=None,
            random_state=seed, n_jobs=-1
        ),
        "Gradient Boosting": GradientBoostingClassifier(random_state=seed),
        "Logistic Regression": LogisticRegression(max_iter=3000, class_weight=None, random_state=seed),
        "Dummy (prior)": DummyClassifier(strategy="prior")
    }

def make_pipeline(model):
    return Pipeline([("preprocess", clone(preprocessor)), ("model", model)])

print("Numeric predictors:", numeric_cols)
print("Categorical predictors:", categorical_cols)
print("Models:", list(make_models(0)))

## 5. Cross-validation: group-aware primary analysis

Primary evaluation uses `StratifiedGroupKFold`: identical predictor-answer patterns are kept in one fold. For each outer fold, the threshold is selected using out-of-fold predictions generated only from the outer-training portion (inner group-aware CV). The selected threshold is then applied to the untouched outer validation fold. This avoids tuning the threshold on the same labels used to evaluate it.

In [ ]:
def safe_auc(y_true, prob):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, prob)

def metric_row(y_true, prob, threshold, model_name, seed, fold, threshold_source):
    pred = (prob >= threshold).astype(int)
    return {
        "model": model_name, "seed": seed, "fold": fold,
        "threshold": float(threshold), "threshold_source": threshold_source,
        "roc_auc": safe_auc(y_true, prob),
        "pr_auc": average_precision_score(y_true, prob),
        "recall": recall_score(y_true, pred, zero_division=0),
        "precision": precision_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "accuracy": accuracy_score(y_true, pred),
        "brier": brier_score_loss(y_true, prob),
        "flagged_fraction": float(pred.mean()),
        "n_validation": int(len(y_true))
    }

def make_group_splits(X_part, y_part, groups_part, n_splits, seed):
    n_groups = len(np.unique(groups_part))
    n_splits_eff = min(n_splits, n_groups)
    if n_splits_eff < 2:
        raise ValueError("Not enough distinct answer-pattern groups for group-aware CV.")
    splitter = StratifiedGroupKFold(n_splits=n_splits_eff, shuffle=True, random_state=seed)
    return list(splitter.split(X_part, y_part, groups_part))

def choose_threshold_inner(X_train, y_train, groups_train, seed, model_name):
    # Inner OOF predictions, used only within the outer training partition.
    splits = make_group_splits(X_train, y_train, groups_train, min(4, N_SPLITS), seed + 1000)
    inner_prob = np.full(len(y_train), np.nan, dtype=float)
    model = make_models(seed)[model_name]
    for itr, iva in splits:
        pipe = make_pipeline(clone(model))
        pipe.fit(X_train.iloc[itr], y_train.iloc[itr])
        inner_prob[iva] = pipe.predict_proba(X_train.iloc[iva])[:, 1]
    ok = np.isfinite(inner_prob)
    if ok.sum() == 0 or len(np.unique(y_train[ok])) < 2:
        return 0.5
    # Maximize F1 over a grid; deterministic tie-break prefers threshold closest to 0.5.
    thresholds = np.linspace(0.05, 0.95, 91)
    scores = [f1_score(y_train[ok], (inner_prob[ok] >= t).astype(int), zero_division=0) for t in thresholds]
    best = np.flatnonzero(np.isclose(scores, np.nanmax(scores)))
    best_idx = min(best, key=lambda i: abs(thresholds[i] - 0.5))
    return float(thresholds[best_idx])

all_metric_rows = []
oof_store = {}  # key=(model, seed); stores out-of-fold probability per row
fold_store = {} # key=(model, seed); stores fold IDs per row
for seed in SEEDS:
    print(f"Running group-aware CV, seed={seed} ...")
    outer_splits = make_group_splits(X, y.to_numpy(), groups, N_SPLITS, seed)
    for model_name, base_model in make_models(seed).items():
        oof_prob = np.full(len(y), np.nan, dtype=float)
        oof_fold = np.full(len(y), -1, dtype=int)
        for fold, (tr, va) in enumerate(outer_splits):
            # Safety check: no predictor-pattern group may cross this split.
            overlap = set(groups[tr]).intersection(set(groups[va]))
            if overlap:
                raise RuntimeError("Group leakage detected: at least one answer-pattern group crosses folds.")
            pipe = make_pipeline(clone(base_model))
            pipe.fit(X.iloc[tr], y.iloc[tr])
            prob = pipe.predict_proba(X.iloc[va])[:, 1]
            oof_prob[va] = prob
            oof_fold[va] = fold

            # Evaluate the same probability predictions at 0.5 and at an inner-CV-selected threshold.
            all_metric_rows.append(metric_row(y.iloc[va].to_numpy(), prob, 0.5, model_name, seed, fold, "fixed_0.5"))
            if model_name != "Dummy (prior)":
                threshold = choose_threshold_inner(X.iloc[tr].reset_index(drop=True),
                                                   y.iloc[tr].reset_index(drop=True),
                                                   np.asarray(groups)[tr], seed + fold, model_name)
            else:
                threshold = 0.5
            all_metric_rows.append(metric_row(y.iloc[va].to_numpy(), prob, threshold, model_name, seed, fold, "inner_group_cv_tuned"))
        oof_store[(model_name, seed)] = oof_prob
        fold_store[(model_name, seed)] = oof_fold

fold_metrics = pd.DataFrame(all_metric_rows)
fold_metrics.to_csv(OUTPUT_DIR / "03_group_cv_fold_metrics.csv", index=False)
display(fold_metrics.head(12).round(4))
print("Saved:", OUTPUT_DIR / "03_group_cv_fold_metrics.csv")

## 6. Paper summary table: mean ± SD across outer folds/seeds

Each reported metric is summarized from fold-level scores (mean and sample SD). Since F1 is calculated separately within each validation fold, mean fold F1 is not generally equal to the harmonic mean of the separately averaged precision and recall. Do not report SD as a 95% confidence interval.

In [ ]:
metric_cols = ["roc_auc", "pr_auc", "recall", "precision", "f1", "accuracy", "brier", "flagged_fraction"]
summary_rows = []
for (model, rule), d in fold_metrics.groupby(["model", "threshold_source"], sort=False):
    row = {"Model": model, "Threshold rule": rule, "n fold results": len(d)}
    for col in metric_cols:
        row[col + "_mean"] = d[col].mean()
        row[col + "_sd"] = d[col].std(ddof=1)
        row[col + "_mean_sd"] = f"{d[col].mean():.3f} ± {d[col].std(ddof=1):.3f}" if d[col].notna().sum() else "NA"
    summary_rows.append(row)
metric_summary = pd.DataFrame(summary_rows)
metric_summary.to_csv(OUTPUT_DIR / "04_paper_metric_summary.csv", index=False)
display(metric_summary[["Model", "Threshold rule", "roc_auc_mean_sd", "pr_auc_mean_sd",
                        "recall_mean_sd", "precision_mean_sd", "f1_mean_sd",
                        "flagged_fraction_mean_sd"]])

## 7. Critical consistency check: tuned versus 0.5 threshold

For a given model/seed/fold, the probabilities are identical between threshold rows. Therefore ROC-AUC and PR-AUC must match exactly. Any difference indicates that different models, folds, or probability predictions were accidentally compared.

In [ ]:
consistency_rows = []
for (model, seed, fold), d in fold_metrics.groupby(["model", "seed", "fold"]):
    a = d[d.threshold_source == "fixed_0.5"]
    b = d[d.threshold_source == "inner_group_cv_tuned"]
    if len(a) and len(b):
        a, b = a.iloc[0], b.iloc[0]
        consistency_rows.append({
            "model": model, "seed": seed, "fold": fold,
            "roc_auc_difference": abs(a.roc_auc - b.roc_auc),
            "pr_auc_difference": abs(a.pr_auc - b.pr_auc),
            "threshold_0.5": a.threshold,
            "tuned_threshold": b.threshold,
            "recall_0.5": a.recall, "recall_tuned": b.recall,
            "precision_0.5": a.precision, "precision_tuned": b.precision,
            "f1_0.5": a.f1, "f1_tuned": b.f1
        })
consistency = pd.DataFrame(consistency_rows)
display(consistency.head(15).round(6))
if len(consistency):
    print("Maximum ROC-AUC difference:", consistency.roc_auc_difference.max())
    print("Maximum PR-AUC difference:", consistency.pr_auc_difference.max())
    assert consistency.roc_auc_difference.max() < 1e-12, "ROC-AUC changed between thresholds: investigate pipeline."
    assert consistency.pr_auc_difference.max() < 1e-12, "PR-AUC changed between thresholds: investigate pipeline."
consistency.to_csv(OUTPUT_DIR / "05_threshold_consistency_check.csv", index=False)

## 8. Group-bootstrap confidence intervals for ROC-AUC and PR-AUC

This is a cluster/group bootstrap over answer-pattern groups using the pooled out-of-fold predictions for each model and seed. It quantifies uncertainty conditional on these fitted out-of-fold predictions; it is not a replacement for external validation. We also compute paired bootstrap intervals for differences in ROC-AUC against Random Forest where possible.

In [ ]:
def bootstrap_group_ci(y_true, prob, group_ids, n_boot=1000, seed=123):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    prob = np.asarray(prob)
    group_ids = np.asarray(group_ids)
    unique_groups = np.unique(group_ids)
    group_rows = {g: np.flatnonzero(group_ids == g) for g in unique_groups}
    aucs, aps = [], []
    for _ in range(n_boot):
        sampled = rng.choice(unique_groups, size=len(unique_groups), replace=True)
        idx = np.concatenate([group_rows[g] for g in sampled])
        yy, pp = y_true[idx], prob[idx]
        if len(np.unique(yy)) < 2:
            continue
        aucs.append(roc_auc_score(yy, pp))
        aps.append(average_precision_score(yy, pp))
    def interval(vals):
        if not vals: return (np.nan, np.nan)
        return tuple(np.quantile(vals, [0.025, 0.975]))
    return interval(aucs), interval(aps)

boot_rows = []
for (model, seed), prob in oof_store.items():
    ok = np.isfinite(prob)
    if ok.sum() == 0: continue
    (auc_lo, auc_hi), (ap_lo, ap_hi) = bootstrap_group_ci(
        y.to_numpy()[ok], prob[ok], np.asarray(groups)[ok],
        n_boot=N_BOOTSTRAP, seed=seed + 900 + sum(map(ord, model))
    )
    boot_rows.append({
        "model": model, "seed": seed,
        "roc_auc_oof": safe_auc(y.to_numpy()[ok], prob[ok]),
        "roc_auc_lo": auc_lo, "roc_auc_hi": auc_hi,
        "pr_auc_oof": average_precision_score(y.to_numpy()[ok], prob[ok]),
        "pr_auc_lo": ap_lo, "pr_auc_hi": ap_hi,
        "bootstrap_replicates_requested": N_BOOTSTRAP
    })
bootstrap_ci = pd.DataFrame(boot_rows)
bootstrap_ci.to_csv(OUTPUT_DIR / "06_group_bootstrap_95CI.csv", index=False)
display(bootstrap_ci.round(4))

## 9. Group-aware permutation importance

Permutation importance is measured as the decrease in average precision (PR-AUC) when one original predictor column is permuted in a held-out fold. This is model reliance, not causality. The mean and SD across folds/seeds are descriptive; overlapping SD/error bars are not a formal pairwise significance test. A feature may have negative importance due to sampling variability.

In [ ]:
perm_rows = []
importance_models = ["Random Forest"]  # change to add models, but this costs more time
for seed in SEEDS:
    splits = make_group_splits(X, y.to_numpy(), groups, N_SPLITS, seed)
    model_dict = make_models(seed)
    for fold, (tr, va) in enumerate(splits):
        for model_name in importance_models:
            pipe = make_pipeline(clone(model_dict[model_name]))
            pipe.fit(X.iloc[tr], y.iloc[tr])
            base_ap = average_precision_score(y.iloc[va], pipe.predict_proba(X.iloc[va])[:, 1])
            rng = np.random.default_rng(seed * 1000 + fold)
            for col in X.columns:
                drops = []
                for rep in range(N_PERM_REPEATS):
                    Xv = X.iloc[va].copy()
                    shuffled = Xv[col].to_numpy(copy=True)
                    rng.shuffle(shuffled)
                    Xv[col] = shuffled
                    p = pipe.predict_proba(Xv)[:, 1]
                    drops.append(base_ap - average_precision_score(y.iloc[va], p))
                perm_rows.append({
                    "model": model_name, "seed": seed, "fold": fold,
                    "feature": col, "baseline_pr_auc": base_ap,
                    "importance_mean": float(np.mean(drops))
                })

perm_long = pd.DataFrame(perm_rows)
perm_summary = (perm_long.groupby(["model", "feature"])["importance_mean"]
                .agg(mean="mean", sd="std", n="count").reset_index()
                .sort_values("mean", ascending=False))
perm_summary["rank_by_point_estimate"] = perm_summary["mean"].rank(ascending=False, method="min").astype(int)
perm_long.to_csv(OUTPUT_DIR / "07_permutation_importance_fold_level.csv", index=False)
perm_summary.to_csv(OUTPUT_DIR / "08_permutation_importance_summary.csv", index=False)
display(perm_summary.round(5))

plt.figure(figsize=(9, max(5, 0.32 * len(perm_summary))))
plot_df = perm_summary.sort_values("mean", ascending=True)
plt.barh(plot_df["feature"], plot_df["mean"], xerr=plot_df["sd"], capsize=2)
plt.axvline(0, linestyle="--")
plt.xlabel("Decrease in PR-AUC after feature permutation")
plt.title("Group-aware permutation importance (mean ± SD across fold/seed results)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_permutation_importance.png", dpi=200, bbox_inches="tight")
plt.show()

## 10. Optional: paired group-bootstrap difference between feature importance values

The following compares each feature's fold-level permutation-importance values with the feature having the highest point estimate using paired fold/seed records. Because fold scores are dependent and the number of folds is small, treat this as exploratory. For a stronger test, retain row-level/group-level permutation draws and bootstrap them under a pre-specified procedure.

In [ ]:
# This descriptive comparison deliberately does NOT label features "significantly different".
if len(perm_summary):
    top_feature = perm_summary.iloc[0]["feature"]
    wide = perm_long.pivot_table(index=["seed", "fold"], columns="feature",
                                 values="importance_mean", aggfunc="mean")
    diff_rows = []
    if top_feature in wide.columns:
        for feature in wide.columns:
            vals = (wide[top_feature] - wide[feature]).dropna()
            diff_rows.append({
                "reference_top_feature": top_feature,
                "feature_compared": feature,
                "mean_paired_difference": vals.mean(),
                "sd_paired_difference": vals.std(ddof=1),
                "n_fold_seed_pairs": len(vals),
                "interpretation": "descriptive only; not a formal significance test"
            })
    importance_differences = pd.DataFrame(diff_rows).sort_values("mean_paired_difference")
    importance_differences.to_csv(OUTPUT_DIR / "09_importance_paired_descriptive_differences.csv", index=False)
    display(importance_differences.round(5))

## 11. Data checks and diagnostics

Inspect these outputs before reporting results. In particular, verify that predictor count, target prevalence, duplicate counts, and conflict-pattern counts match the manuscript's data description. If they do not, do not manually edit the output to force a match; investigate the CSV, missing-value handling, excluded columns, and target encoding.

In [ ]:
# Fold balance and unique-group audit
fold_audit_rows = []
for seed in SEEDS:
    splits = make_group_splits(X, y.to_numpy(), groups, N_SPLITS, seed)
    for fold, (tr, va) in enumerate(splits):
        fold_audit_rows.append({
            "seed": seed, "fold": fold,
            "train_n": len(tr), "validation_n": len(va),
            "train_at_risk_prevalence": float(y.iloc[tr].mean()),
            "validation_at_risk_prevalence": float(y.iloc[va].mean()),
            "train_groups": len(np.unique(np.asarray(groups)[tr])),
            "validation_groups": len(np.unique(np.asarray(groups)[va])),
            "group_overlap": len(set(np.asarray(groups)[tr]).intersection(set(np.asarray(groups)[va])))
        })
fold_audit = pd.DataFrame(fold_audit_rows)
display(fold_audit)
assert (fold_audit["group_overlap"] == 0).all()
fold_audit.to_csv(OUTPUT_DIR / "10_fold_audit.csv", index=False)

# Basic target prevalence baseline for PR-AUC context
print(f"At-risk prevalence / no-skill PR-AUC reference: {y.mean():.4f}")
print("Number of answer-pattern groups:", len(np.unique(groups)))
print("Number of modelling rows:", len(y))

## 12. Export a compact results bundle and environment record

In [ ]:
# Save environment information and download ALL generated outputs in one ZIP.
# Every CSV, PNG, JSON, and other file inside OUTPUT_DIR will be included.
import shutil

NEW = Path(OUTPUT_DIR)
NEW.mkdir(parents=True, exist_ok=True)

env = {
    "python": sys.version,
    "platform": platform.platform(),
    "pandas": pd.__version__,
}
try:
    import sklearn
    env["scikit_learn"] = sklearn.__version__
except Exception:
    pass
try:
    env["numpy"] = np.__version__
except Exception:
    pass
with open(NEW / "environment.json", "w", encoding="utf-8") as f:
    json.dump(env, f, indent=2)

# Collect files recursively so PNGs/CSVs in subfolders are included too.
output_files = sorted(p for p in NEW.rglob("*") if p.is_file())
print(f"Files included in ZIP ({len(output_files)}):")
for p in output_files:
    print(" •", p.relative_to(NEW))

zip_path = Path(shutil.make_archive("outputs_step2", "zip", root_dir=NEW))
print("Created ZIP:", zip_path.resolve())

# Automatically download in Google Colab; in local Jupyter, the ZIP remains in the working folder.
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print("Not running in Google Colab. ZIP saved at:", zip_path.resolve())


## 13. Manuscript language — use only after checking actual outputs

**Sample description template:** “The dataset contained [N] usable responses forming [G] distinct predictor-answer patterns. We kept each answer-pattern group entirely within one validation fold to reduce leakage from repeated response patterns. These groups are not necessarily unique individuals.”

**Metrics template:** “Performance is reported as mean ± standard deviation across outer validation folds and random seeds. Threshold selection was performed using group-aware cross-validation within each outer training partition. ROC-AUC and PR-AUC are unchanged by threshold choice because they are computed from the same predicted probabilities.”

**Importance template:** “The five highest point estimates were [features]. The ranking is descriptive; fold/seed variability means the estimates should not be interpreted as proof that the features are statistically distinguishable. Permutation importance reflects model reliance, not causal effects.”

**Ethics and availability:** The notebook cannot verify IRB approval, consent language, dataset license, or permissions from the CSV itself. Check the original dataset documentation and cite the correct source. Do not claim a repository is public unless you actually publish it; otherwise use a truthful availability statement such as “Code is available from the corresponding author on reasonable request,” if that is accurate.